# Entrenamiento de los cultivos principales

Selecciona cultivos por `Idcultivo` y superficie cosechada entre los registros aptos del dataset de modelado. Para cada cultivo entrena un modelo con clima, suelo e historial productivo calculado exclusivamente con años anteriores.

In [7]:

import pandas as pd
from sklearn.base import clone

from proyecto_agronomia.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from modeling.cultivos import (
    VARIABLES_CATEGORICAS, VARIABLES_NUMERICAS, catalogar_cultivos,
    cargar_registros_cultivos, preparar_cultivo, seleccionar_principales, separar_xy,
)
from modeling.registry import guardar_modelo, ruta_version_modelo
from modeling.train import crear_particiones, seleccionar_modelo
from modeling.train_original import comparar_modelos_originales

In [8]:
RUTA_DATOS = PROCESSED_DATA_DIR / 'agricultura_modelado_2003_2025.csv'
CANTIDAD_CULTIVOS = 385  # maximo; los filtros determinan cuantos son elegibles
UNIDAD = 'Tonelada'
MINIMO_ANIOS = 15
MINIMO_MUNICIPIOS = 100
ANIO_INICIO_PRUEBA = 2022
PROPORCION_MUNICIPIOS_RESERVADOS = 0.20
SEMILLA = 42
VERSION = 'v2_modelado'
ENTRENAR = True

if not RUTA_DATOS.is_file():
    raise FileNotFoundError(f'No se encontro el dataset de entrenamiento: {RUTA_DATOS}')
print(f'Dataset de entrenamiento: {RUTA_DATOS}')

Dataset de entrenamiento: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_modelado_2003_2025.csv


## Catálogo y selección transparente

In [9]:
catalogo = catalogar_cultivos(ruta=RUTA_DATOS, unidad=UNIDAD)
principales = seleccionar_principales(
    catalogo, cantidad=CANTIDAD_CULTIVOS, minimo_anios=MINIMO_ANIOS,
    minimo_municipios=MINIMO_MUNICIPIOS,
)
principales.insert(0, 'posicion', range(1, len(principales) + 1))
display(principales)
assert 0 < len(principales) <= CANTIDAD_CULTIVOS
assert principales['anios'].ge(MINIMO_ANIOS).all()
assert principales['municipios'].ge(MINIMO_MUNICIPIOS).all()
print(f'Cultivos elegibles: {len(principales):,} de hasta {CANTIDAD_CULTIVOS:,} solicitados')

,posicion,Idcultivo,Nomcultivo,superficie_cosechada,registros,anios,municipios
0,1,7490000,Maíz grano,1.609257e+08,105079,23,2432
1,2,15050000,Pastos y praderas,5.006176e+07,28762,21,1128
2,3,8810000,Sorgo grano,3.649216e+07,23228,23,823
3,4,6840000,Frijol,3.418383e+07,64932,23,2143
4,5,5820000,Caña de azúcar,1.702829e+07,7707,23,328
...,...,...,...,...,...,...,...
80,81,7170000,Hortalizas,5.650643e+04,1758,17,155
81,82,8490000,Rábano,5.449668e+04,2606,21,157
82,83,9320000,Zempoalxochitl,3.527392e+04,1715,23,141
83,84,7540000,Mamey,2.848560e+04,1660,21,113


Cultivos elegibles: 85 de hasta 385 solicitados


## Cargar una sola vez los registros seleccionados

La lectura por bloques evita cargar los 825 mil registros completos y evita releer el archivo para cada cultivo.

In [10]:
ids_cultivos = principales['Idcultivo'].astype(int).tolist()
cultivos_entrenamiento = principales[['Idcultivo', 'Nomcultivo']].copy()
assert cultivos_entrenamiento['Idcultivo'].is_unique
assert cultivos_entrenamiento['Nomcultivo'].is_unique
datos_principales = cargar_registros_cultivos(
    ids_cultivos, ruta=RUTA_DATOS, unidad=UNIDAD,
)
display(datos_principales.groupby(['Idcultivo', 'Nomcultivo_canonico']).agg(
    registros=('cvegeo', 'size'), municipios=('cvegeo', 'nunique'),
    anio_inicial=('Anio', 'min'), anio_final=('Anio', 'max'),
).sort_values('registros', ascending=False))

,,registros,municipios,anio_inicial,anio_final
Idcultivo,Nomcultivo_canonico,,,,
7490000,Maíz grano,105079,2432,2003,2025
6840000,Frijol,64932,2143,2003,2025
5490000,Avena forrajera en verde,30392,1041,2003,2025
15050000,Pastos y praderas,28762,1128,2003,2025
8970000,Tomate rojo,25461,1313,2003,2025
...,...,...,...,...,...
8770000,Sorgo escobero,1155,107,2003,2025
9040000,Trigo forrajero verde,828,149,2003,2025
5810000,Canola,734,145,2003,2025


## Entrenamiento reanudable

Cada modelo terminado se guarda inmediatamente. Si se interrumpe la ejecución, al correr de nuevo se omiten los artefactos ya existentes.

In [11]:
resumen_modelos = []
if ENTRENAR:
    for numero, fila in enumerate(cultivos_entrenamiento.itertuples(index=False), start=1):
        idcultivo = int(fila.Idcultivo)
        cultivo = fila.Nomcultivo
        destino = ruta_version_modelo(cultivo, VERSION)
        if (destino / 'model.joblib').is_file():
            print(f'[{numero}/{len(cultivos_entrenamiento)}] Omitido, ya existe: {cultivo}')
            continue
        print(f'[{numero}/{len(cultivos_entrenamiento)}] Entrenando: {cultivo} ({idcultivo})')
        datos = preparar_cultivo(datos_principales, idcultivo)
        particiones = crear_particiones(
            datos, anio_prueba=ANIO_INICIO_PRUEBA,
            proporcion_municipios=PROPORCION_MUNICIPIOS_RESERVADOS, semilla=SEMILLA,
        )
        entrenados, metricas = comparar_modelos_originales(datos, particiones, SEMILLA)
        ganador = seleccionar_modelo(metricas)
        X, y = separar_xy(datos)
        modelo_final = clone(entrenados[ganador]).fit(X, y)
        guardar_modelo(
            modelo=modelo_final, cultivo=cultivo, version=VERSION,
            metadata={
                'algoritmo': ganador, 'idcultivo': idcultivo, 'unidad': UNIDAD,
                'dataset': RUTA_DATOS.name,
                'periodo_datos': f"{datos['Anio'].min()}-{datos['Anio'].max()}",
                'inicio_prueba': ANIO_INICIO_PRUEBA,
                'proporcion_municipios_reservados': PROPORCION_MUNICIPIOS_RESERVADOS,
                'semilla': SEMILLA, 'registros': len(datos),
                'municipios': datos['cvegeo'].nunique(),
                'objetivo': 'Rendimiento_modelado',
                'seleccion': 'Idcultivo, registros aptos y superficie cosechada',
            },
            metricas={'comparacion_modelos': metricas.to_dict(orient='records')},
            variables=[*VARIABLES_NUMERICAS, *VARIABLES_CATEGORICAS],
        )
        mejores = metricas.loc[metricas['modelo'].eq(ganador)]
        for fila in mejores.to_dict(orient='records'):
            resumen_modelos.append({
                'idcultivo': idcultivo, 'cultivo': cultivo,
                'ganador': ganador, **fila,
            })
        print(f'  Guardado {ganador} en {destino}')
else:
    print('Entrenamiento omitido.')

[1/85] Entrenando: Maíz grano (7490000)
  Guardado hist_gradient_boosting en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\maiz_grano\v2_modelado
[2/85] Entrenando: Pastos y praderas (15050000)
  Guardado hist_gradient_boosting en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\pastos_y_praderas\v2_modelado
[3/85] Entrenando: Sorgo grano (8810000)
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\sorgo_grano\v2_modelado
[4/85] Entrenando: Frijol (6840000)
  Guardado hist_gradient_boosting en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\frijol\v2_modelado
[5/85] Entrenando: Caña de azúcar (5820000)
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\cana_de_azucar\v2_modelado
[6/85] Entrenando: Café cereza (5710000)
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\cafe_cereza\v2_modelado
[7/85] Entrenando: Trigo grano (9050000)
  Guardado random_forest en B:\MCD2\JW\agro

b:\MCD2\JW\agro\proyecto_agronomia\.venv\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
b:\MCD2\JW\agro\proyecto_agronomia\.venv\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
b:\MCD2\JW\agro\proyecto_agronomia\.venv\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\hortalizas\v2_modelado
[82/85] Entrenando: Rábano (8490000)
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\rabano\v2_modelado
[83/85] Entrenando: Zempoalxochitl (9320000)
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\zempoalxochitl\v2_modelado
[84/85] Entrenando: Mamey (7540000)
  Guardado hist_gradient_boosting en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\mamey\v2_modelado
[85/85] Entrenando: Granada (7010000)
  Guardado random_forest en B:\MCD2\JW\agro\proyecto_agronomia\models\rendimiento\granada\v2_modelado


## Resumen de modelos entrenados en esta ejecución

In [12]:
df_resumen_modelos = pd.DataFrame(resumen_modelos)
if not df_resumen_modelos.empty:
    display(df_resumen_modelos.sort_values(['particion', 'rmse']))
    destino_resumen = INTERIM_DATA_DIR / 'modeling' / 'cultivos_principales_metricas.csv'
    destino_resumen.parent.mkdir(parents=True, exist_ok=True)
    df_resumen_modelos.to_csv(destino_resumen, index=False)
    print(f'Resumen guardado: {destino_resumen}')
else:
    print('No hubo modelos nuevos; los existentes fueron respetados.')

,idcultivo,cultivo,ganador,modelo,particion,registros,mae,rmse,r2,sesgo
103,7220000,Jamaica,random_forest,random_forest,prueba_espaciotemporal,69,0.078709,0.105991,0.712816,0.019805
7,6840000,Frijol,hist_gradient_boosting,hist_gradient_boosting,prueba_espaciotemporal,2140,0.108063,0.178645,0.859758,-0.004024
47,5080000,Ajonjolí,random_forest,random_forest,prueba_espaciotemporal,154,0.098418,0.196061,0.425157,-0.036196
37,6900000,Garbanzo grano,random_forest,random_forest,prueba_espaciotemporal,184,0.170948,0.254575,0.807887,0.010902
11,5710000,Café cereza,random_forest,random_forest,prueba_espaciotemporal,416,0.152844,0.278257,0.868537,-0.071075
...,...,...,...,...,...,...,...,...,...,...
86,5050000,Agave,random_forest,random_forest,prueba_temporal,1079,7.823075,13.101998,0.504944,-3.182161
94,7330000,Lechuga,hist_gradient_boosting,hist_gradient_boosting,prueba_temporal,815,3.105054,14.497856,0.586583,-1.159009
100,8210000,Pepino,random_forest,random_forest,prueba_temporal,1475,9.024283,18.442437,0.869572,-1.171156
58,8970000,Tomate rojo,random_forest,random_forest,prueba_temporal,3849,11.105938,23.766187,0.927839,-3.099935


Resumen guardado: B:\MCD2\JW\agro\proyecto_agronomia\data\interim\modeling\cultivos_principales_metricas.csv
